# 🔄 Основные рабочие процессы агента с Microsoft Foundry (Python)

## 📋 Руководство по оркестрации рабочих процессов

В этой тетрадке представлены мощные возможности **Workflow Builder** в Microsoft Agent Framework. Узнайте, как создавать сложные многоэтапные рабочие процессы агента, которые могут обрабатывать сложные бизнес-процессы и беспрепятственно координировать несколько операций ИИ.

> **Примечание по миграции:** В этом примере ранее использовались модели GitHub. GitHub Models устарели (выводятся из эксплуатации в июле 2026 года), поэтому теперь используется **Microsoft Foundry** через `FoundryChatClient`, который ориентирован на Azure OpenAI **Responses API**.

## 🎯 Цели обучения

### 🏗️ **Архитектура рабочего процесса**
- **Workflow Builder**: проектирование и оркестрация сложных многоэтапных процессов
- **Событийное выполнение**: обработка событий рабочего процесса и переходов состояний
- **Визуальное проектирование рабочего процесса**: создание и визуализация структуры рабочего процесса
- **Интеграция Microsoft Foundry**: использование моделей ИИ в контексте рабочих процессов

### 🔄 **Оркестрация процессов**
- **Последовательные операции**: выполнение нескольких задач агента в логическом порядке
- **Условная логика**: реализация точек принятия решений и ветвящихся рабочих процессов
- **Обработка ошибок**: надежное восстановление после ошибок и устойчивость рабочего процесса
- **Управление состоянием**: отслеживание и управление состоянием выполнения рабочего процесса

### 📊 **Шаблоны рабочих процессов для предприятий**
- **Автоматизация бизнес-процессов**: автоматизация сложных организационных рабочих процессов
- **Координация многократных агентов**: координация нескольких специализированных агентов
- **Масштабируемое выполнение**: проектирование рабочих процессов для корпоративных операций в больших масштабах
- **Мониторинг и наблюдаемость**: отслеживание эффективности и результатов рабочих процессов

## ⚙️ Требования и настройка

### 📦 **Необходимые зависимости**

Установите Agent Framework с возможностями рабочих процессов:

```bash
pip install agent-framework -U
```

### 🔑 **Конфигурация Microsoft Foundry**

Войдите в систему с помощью Azure CLI (`az login`), чтобы `AzureCliCredential` мог аутентифицироваться, затем укажите данные вашего проекта Microsoft Foundry.

**Настройка окружения (.env файл):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Корпоративные сценарии использования**

**Примеры бизнес-процессов:**
- **Введение клиентов**: многоэтапная проверка и процессы настройки
- **Поток контента**: автоматизированное создание, проверка и публикация контента
- **Обработка данных**: ETL рабочие процессы с преобразованием на основе ИИ
- **Контроль качества**: автоматизированное тестирование и проверка

**Преимущества рабочих процессов:**
- 🎯 **Надежность**: детерминированное выполнение с восстановлением после ошибок
- 📈 **Масштабируемость**: обработка автоматизации процессов с большим объёмом
- 🔍 **Наблюдаемость**: полные аудиторские следы и мониторинг
- 🔧 **Поддерживаемость**: визуальное проектирование и модульные компоненты

## 🎨 Шаблоны проектирования рабочих процессов

### Основная структура рабочего процесса
```mermaid
graph TD
    A[Начало] --> B[Задача агента 1]
    B --> C{Точка принятия решения}
    C -->|Успех| D[Задача агента 2]
    C -->|Неудача| E[Обработчик ошибок]
    D --> F[Конец]
    E --> F
```

**Ключевые компоненты:**
- **WorkflowBuilder**: основной движок оркестрации
- **WorkflowEvent**: обработка событий и коммуникация
- **WorkflowViz**: визуальное представление рабочего процесса и отладка

Давайте создадим ваш первый интеллектуальный рабочий процесс! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
